# Classification d'images avec CIFAR-10

Ce notebook explore différentes architectures de réseaux de neurones appliquées au dataset CIFAR-10. On part des modèles les plus simples et on monte progressivement en complexité pour voir ce que ça change réellement.

In [ ]:
# imports de base
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

print("TensorFlow version:", tf.__version__)
print("GPU disponible:", tf.config.list_physical_devices('GPU'))

## 1. Le dataset CIFAR-10

CIFAR-10 (Canadian Institute For Advanced Research) contient 60 000 images couleur de 32x32 pixels réparties en 10 classes : avion, voiture, oiseau, chat, cerf, chien, grenouille, cheval, bateau, camion. 50 000 images sont pour l'entraînement et 10 000 pour le test.

C'est un dataset classique pour tester des architectures de classification. Les images sont assez petites et la tâche est loin d'être triviale — distinguer un chat d'un chien sur 32x32 pixels c'est pas évident même pour nous parfois.

In [ ]:
# chargement
cifar10 = tf.keras.datasets.cifar10
(train_images, train_labels), (test_images, test_labels) = cifar10.load_data()

class_names = ['Airplane', 'Automobile', 'Bird', 'Cat', 'Deer',
               'Dog', 'Frog', 'Horse', 'Ship', 'Truck']

print("Train:", train_images.shape, "|", len(train_labels), "labels")
print("Test: ", test_images.shape,  "|", len(test_labels),  "labels")
print("Valeurs pixel min/max:", train_images.min(), train_images.max())

In [ ]:
# afficher quelques exemples pour voir à quoi ça ressemble
plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(train_images[i])
    plt.title(class_names[train_labels[i][0]], fontsize=8)
    plt.axis('off')
plt.suptitle("Quelques exemples du dataset", fontsize=11)
plt.tight_layout()
plt.show()

### Normalisation

On divise les valeurs pixel par 255 pour les ramener dans [0, 1]. C'est important pour l'entraînement : les gradients sont plus stables et la convergence est plus rapide. On applique la même transformation au train et au test — c'est obligatoire, le réseau doit voir les mêmes distributions.

In [ ]:
train_images = train_images / 255.0
test_images  = test_images  / 255.0

## 2. Comprendre les couches

Avant de lancer des modèles, prenons le temps d'expliquer les briques de base qu'on va utiliser.

**Flatten** : transforme un volume 3D (32×32×3) en vecteur 1D (3072 valeurs). C'est juste un aplatissement, aucun paramètre appris. Indispensable pour connecter une partie convolutive à des couches denses.

**Dense (couche fully-connected)** : chaque neurone est connecté à tous les neurones de la couche précédente. Si on a 128 neurones et une entrée de taille 3072, on a 3072×128 + 128 = 393 344 paramètres rien que pour cette couche. C'est beaucoup, et ça ne tient pas compte de la structure spatiale de l'image.

**Conv2D** : applique un filtre (petit noyau, typiquement 3×3 ou 5×5) qui glisse sur toute l'image. Le même filtre est partagé sur toute la surface — c'est ça l'idée forte : un détecteur de bord appris en haut de l'image marche aussi en bas. On a beaucoup moins de paramètres qu'un Dense, et on exploite la localité des patterns visuels.

Illustration du principe : un filtre 3×3 calcule une combinaison linéaire des 9 pixels voisins à chaque position, puis passe le résultat dans une activation (ReLU en général). On obtient une "feature map" qui indique où ce motif est présent dans l'image.

**MaxPooling2D** : réduit la résolution spatiale en gardant le maximum sur une fenêtre (souvent 2×2). Ça rend le réseau moins sensible aux petits décalages et réduit la taille des représentations.

**BatchNormalization** : normalise les activations au sein d'un batch. Aide beaucoup la convergence, surtout sur des réseaux profonds, en réduisant le problème de covariate shift interne.

**Dropout** : éteint aléatoirement une fraction des neurones pendant l'entraînement. Force le réseau à ne pas trop dépendre d'un seul neurone. Technique de régularisation efficace contre l'overfitting.

**Activation ReLU** : f(x) = max(0, x). Simple, efficace, évite le problème de gradient qui disparaît (vanishing gradient) mieux que tanh ou sigmoid. C'est la valeur par défaut dans la plupart des réseaux modernes.

Sources : LeCun et al., "Gradient-based learning applied to document recognition", 1998 (LeNet). Krizhevsky et al., "ImageNet Classification with Deep CNNs", 2012 (AlexNet). Goodfellow et al., Deep Learning, MIT Press, 2016.

## 3. Les modèles

On va tester 5 architectures, du plus simple au plus sophistiqué. L'idée est de voir comment la complexité du modèle impacte les performances.

In [ ]:
# modèle le plus simple possible : juste un Flatten + une Dense de sortie
# c'est une régression logistique multivariée en gros
def create_simplest_cifar10_model():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(32, 32, 3)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(10)
    ], name="Simplest_NN")
    return model

# on ajoute une couche cachée avec ReLU — c'est le réseau "vanilla"
def create_vanilla_nn_cifar10_model():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(32, 32, 3)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(10)
    ], name="Vanilla_NN")
    return model

# LeNet-5 adapté à CIFAR-10 (originalement fait pour MNIST)
# architecture convolutive classique des années 90
def create_lenet5_cifar10_model():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(32, 32, 3)),
        tf.keras.layers.Conv2D(6, kernel_size=(5, 5), activation='tanh'),
        tf.keras.layers.AveragePooling2D(pool_size=(2, 2)),
        tf.keras.layers.Conv2D(16, kernel_size=(5, 5), activation='tanh'),
        tf.keras.layers.AveragePooling2D(pool_size=(2, 2)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(120, activation='tanh'),
        tf.keras.layers.Dense(84, activation='tanh'),
        tf.keras.layers.Dense(10)
    ], name="LeNet5")
    return model

# CNN plus moderne avec 3 blocs conv+pooling et ReLU
def create_vanilla_cnn_cifar10_model():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(32, 32, 3)),
        tf.keras.layers.Conv2D(32, kernel_size=(3, 3), activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
        tf.keras.layers.Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
        tf.keras.layers.Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(10)
    ], name="Vanilla_CNN")
    return model

# AlexNet adapté aux images 32x32 — architecture de 2012 qui a révolutionné ImageNet
# on garde la structure mais on adapte aux petites images
def create_alexnet_cifar10_model():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(32, 32, 3)),
        tf.keras.layers.Conv2D(96, kernel_size=(3, 3), activation='relu', padding='same'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(pool_size=(2, 2), strides=(2, 2)),
        tf.keras.layers.Conv2D(256, kernel_size=(3, 3), activation='relu', padding='same'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(pool_size=(2, 2), strides=(2, 2)),
        tf.keras.layers.Conv2D(384, kernel_size=(3, 3), activation='relu', padding='same'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Conv2D(384, kernel_size=(3, 3), activation='relu', padding='same'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Conv2D(256, kernel_size=(3, 3), activation='relu', padding='same'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(pool_size=(2, 2), strides=(2, 2)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(4096, activation='relu'),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(4096, activation='relu'),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(10)
    ], name="AlexNet")
    return model

# créer les 5 modèles
model_simplest   = create_simplest_cifar10_model()
model_vanilla_nn = create_vanilla_nn_cifar10_model()
model_lenet5     = create_lenet5_cifar10_model()
model_vanilla_cnn = create_vanilla_cnn_cifar10_model()
model_alexnet    = create_alexnet_cifar10_model()

# voir un résumé du vanilla CNN pour avoir une idée des dimensions
model_vanilla_cnn.summary()

## 4. Paramètres d'apprentissage

Avant d'entraîner, expliquons les hyperparamètres qu'on utilise.

**Learning rate (taux d'apprentissage)** : contrôle la taille des pas lors de la descente de gradient. Trop grand = le modèle oscille et diverge. Trop petit = convergence très lente. 0.001 est une valeur de départ raisonnable pour Adam.

**Optimiseur Adam** : combine les avantages de RMSprop et du momentum. Il adapte le learning rate pour chaque paramètre individuellement. beta_1 et beta_2 sont les coefficients de décroissance des moyennes mobiles (moments). En pratique on touche rarement à ces valeurs.

**Epochs** : nombre de passages complets sur tout le dataset d'entraînement. 10 c'est peu pour CIFAR-10, mais suffisant pour comparer les modèles entre eux.

**Batch size** : nombre d'images traitées avant chaque mise à jour des poids. 64 est un bon compromis : assez grand pour estimer correctement le gradient, assez petit pour ne pas exploser la mémoire GPU.

**Loss SparseCategoricalCrossentropy** : mesure l'écart entre la distribution prédite et le vrai label. `from_logits=True` signifie que la dernière couche ne passe pas par une softmax — la loss s'en charge directement, ce qui est numériquement plus stable.

**Pourquoi évaluer sur le test et pas sur le train ?** Le modèle optimise ses poids sur les données d'entraînement. Il peut très bien "mémoriser" celles-ci et atteindre 99% d'accuracy en train tout en étant mauvais sur des données nouvelles (overfitting). La précision sur le jeu de test est la seule mesure qui compte vraiment : elle reflète la capacité du modèle à généraliser.

In [ ]:
# fonction utilitaire pour compiler, entraîner et évaluer un modèle
def train_model(model, epochs=10, batch_size=64, lr=0.001):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=['accuracy']
    )
    history = model.fit(
        train_images, train_labels,
        batch_size=batch_size,
        epochs=epochs,
        validation_data=(test_images, test_labels),
        verbose=1
    )
    return history

# petite fonction pour afficher les courbes d'entraînement
def plot_history(history, title=''):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    ax1.plot(history.history['accuracy'],     label='train')
    ax1.plot(history.history['val_accuracy'], label='test')
    ax1.set_title('Accuracy' + (f' - {title}' if title else ''))
    ax1.set_xlabel('Epoch')
    ax1.legend()
    
    ax2.plot(history.history['loss'],     label='train')
    ax2.plot(history.history['val_loss'], label='test')
    ax2.set_title('Loss' + (f' - {title}' if title else ''))
    ax2.set_xlabel('Epoch')
    ax2.legend()
    
    plt.tight_layout()
    plt.show()

## 5. Entraînement des modèles

On entraîne les 5 modèles avec les mêmes hyperparamètres pour une comparaison équitable.

In [ ]:
print("=== Simplest NN ===")
history_simplest = train_model(model_simplest, epochs=10)
plot_history(history_simplest, 'Simplest NN')

In [ ]:
print("=== Vanilla NN ===")
history_vanilla_nn = train_model(model_vanilla_nn, epochs=10)
plot_history(history_vanilla_nn, 'Vanilla NN')

In [ ]:
print("=== LeNet-5 ===")
history_lenet5 = train_model(model_lenet5, epochs=10)
plot_history(history_lenet5, 'LeNet-5')

In [ ]:
print("=== Vanilla CNN ===")
history_vanilla_cnn = train_model(model_vanilla_cnn, epochs=10)
plot_history(history_vanilla_cnn, 'Vanilla CNN')

In [ ]:
print("=== AlexNet ===")
history_alexnet = train_model(model_alexnet, epochs=10)
plot_history(history_alexnet, 'AlexNet')

## 6. Synthèse des résultats

On récupère les métriques finales sur le test set pour tous les modèles.

In [ ]:
# récupérer les accuracy test de chaque modèle
models_dict = {
    'Simplest NN':  (model_simplest,   history_simplest),
    'Vanilla NN':   (model_vanilla_nn,  history_vanilla_nn),
    'LeNet-5':      (model_lenet5,      history_lenet5),
    'Vanilla CNN':  (model_vanilla_cnn, history_vanilla_cnn),
    'AlexNet':      (model_alexnet,     history_alexnet),
}

print(f"{'Modèle':<20} {'Params':>10} {'Test Acc':>10}")
print('-' * 43)
for name, (m, h) in models_dict.items():
    acc  = max(h.history['val_accuracy'])
    params = m.count_params()
    print(f"{name:<20} {params:>10,} {acc:>10.3f}")

In [ ]:
# courbes comparatives des val accuracy
plt.figure(figsize=(10, 5))
for name, (m, h) in models_dict.items():
    plt.plot(h.history['val_accuracy'], label=name)
plt.title('Accuracy sur le test set — comparaison des modèles')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

On voit clairement que les CNNs surpassent les réseaux denses. C'est logique : les couches convolutives exploitent la structure spatiale des images (un pixel est corrélé à ses voisins) là où un Dense traite chaque pixel indépendamment.

## 7. Matrice de confusion

In [ ]:
# on fait la matrice de confusion sur le meilleur modèle (Vanilla CNN ou AlexNet)
# utilise le modèle vanilla_cnn comme référence

def plot_confusion_matrix(model, model_name):
    preds = model.predict(test_images)
    predicted_classes = np.argmax(preds, axis=1)
    true_classes = test_labels.flatten()
    
    cm = confusion_matrix(true_classes, predicted_classes)
    cm_pct = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
    
    fig, ax = plt.subplots(figsize=(9, 7))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm_pct,
                                  display_labels=class_names)
    disp.plot(cmap='Blues', values_format='.1f', ax=ax)
    plt.title(f'Matrice de confusion (%) — {model_name}')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

plot_confusion_matrix(model_vanilla_cnn, 'Vanilla CNN')

La diagonale représente les prédictions correctes. On s'attend à voir les plus grosses confusions entre des classes visuellement proches : chat/chien, voiture/camion, avion/oiseau par exemple.

## 8. Influence des hyperparamètres

On va maintenant tester l'impact du learning rate et du batch size sur les performances, en prenant le Vanilla CNN comme modèle de référence.

In [ ]:
# test de différents learning rates
learning_rates = [0.01, 0.001, 0.0001]
lr_results = {}

for lr in learning_rates:
    print(f"\nLearning rate = {lr}")
    m = create_vanilla_cnn_cifar10_model()
    h = train_model(m, epochs=10, lr=lr)
    lr_results[lr] = h

# comparer
plt.figure(figsize=(8, 5))
for lr, h in lr_results.items():
    plt.plot(h.history['val_accuracy'], label=f'lr={lr}')
plt.title('Impact du learning rate (Vanilla CNN)')
plt.xlabel('Epoch')
plt.ylabel('Val Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# test de différents batch sizes
batch_sizes = [32, 64, 256]
bs_results = {}

for bs in batch_sizes:
    print(f"\nBatch size = {bs}")
    m = create_vanilla_cnn_cifar10_model()
    h = train_model(m, epochs=10, batch_size=bs)
    bs_results[bs] = h

plt.figure(figsize=(8, 5))
for bs, h in bs_results.items():
    plt.plot(h.history['val_accuracy'], label=f'bs={bs}')
plt.title('Impact du batch size (Vanilla CNN)')
plt.xlabel('Epoch')
plt.ylabel('Val Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

Un batch trop grand peut donner une convergence moins bonne (le gradient est trop lissé, on saute par-dessus des minima locaux intéressants). Un batch trop petit entraîne beaucoup de bruit dans les mises à jour. 64-128 c'est généralement le sweet spot pour CIFAR-10.

## 9. Propositions d'améliorations

Pour améliorer les scores, plusieurs pistes :

**Data augmentation** : on génère de nouvelles images à partir des existantes (flip horizontal, rotation légère, zoom). Le modèle voit ainsi des variantes qu'il n'a pas en mémoire, ce qui réduit l'overfitting.

**Architectures plus profondes** : VGG, ResNet, etc. Les connexions résiduelles de ResNet permettent d'entraîner des réseaux très profonds sans que le gradient disparaisse.

**Régularisation supplémentaire** : L2 sur les poids, plus de Dropout.

**Learning rate scheduling** : commencer avec un grand LR et le réduire au fil des epochs.

In [ ]:
# CNN amélioré avec data augmentation et BatchNorm

def create_improved_cnn():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(32, 32, 3)),
        
        # data augmentation en ligne (seulement pendant l'entraînement)
        tf.keras.layers.RandomFlip('horizontal'),
        tf.keras.layers.RandomRotation(0.1),
        tf.keras.layers.RandomZoom(0.1),
        
        # bloc 1
        tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(2),
        tf.keras.layers.Dropout(0.2),
        
        # bloc 2
        tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(2),
        tf.keras.layers.Dropout(0.3),
        
        # bloc 3
        tf.keras.layers.Conv2D(128, 3, padding='same', activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Conv2D(128, 3, padding='same', activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(2),
        tf.keras.layers.Dropout(0.4),
        
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(10)
    ], name="Improved_CNN")
    return model

model_improved = create_improved_cnn()
model_improved.summary()

In [ ]:
# learning rate qui diminue au fil des epochs
lr_schedule = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_accuracy', factor=0.5, patience=3, verbose=1
)

model_improved.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

history_improved = model_improved.fit(
    train_images, train_labels,
    batch_size=64,
    epochs=30,
    validation_data=(test_images, test_labels),
    callbacks=[lr_schedule],
    verbose=1
)

plot_history(history_improved, 'Improved CNN')

In [ ]:
# score final du modèle amélioré
test_loss, test_acc = model_improved.evaluate(test_images, test_labels, verbose=0)
print(f"Test accuracy modèle amélioré : {test_acc:.4f}")

## 10. Explicabilité avec GradCAM++

GradCAM (Gradient-weighted Class Activation Mapping) permet de visualiser quelles parties de l'image ont influencé la décision du réseau. L'idée : on calcule le gradient de la prédiction par rapport aux activations de la dernière couche convolutive, et on les pondère pour créer une carte de chaleur.

GradCAM++ est une version améliorée qui donne de meilleures localisations notamment quand plusieurs objets de même classe sont présents (Chattopadhay et al., 2018).

On utilise la librairie tf-keras-vis qui implémente ça proprement.

In [ ]:
# installation de tf-keras-vis
!pip install tf-keras-vis -q

In [ ]:
from tf_keras_vis.gradcam_plus_plus import GradcamPlusPlus
from tf_keras_vis.utils.scores import CategoricalScore
from tf_keras_vis.utils.model_modifiers import ReplaceToLinear
from matplotlib import cm as mpl_cm

# on travaille sur le vanilla CNN (qui a des couches conv bien identifiables)
# on prend quelques images de test pour visualiser
sample_indices = [0, 5, 10, 15, 20, 30]  # indices dans test_images
sample_images = test_images[sample_indices]
sample_labels = test_labels[sample_indices].flatten()

# prédictions du modèle sur ces images
preds = model_vanilla_cnn.predict(sample_images)
pred_classes = np.argmax(preds, axis=1)

# score : on s'intéresse à la classe prédite
score = CategoricalScore(pred_classes.tolist())

# GradCAM++ — replace_to_linear enlève la softmax pour le calcul des gradients
gradcam = GradcamPlusPlus(model_vanilla_cnn, model_modifier=ReplaceToLinear(), clone=True)
cam = gradcam(score, sample_images, penultimate_layer=-1)

# affichage
fig, axes = plt.subplots(2, len(sample_indices), figsize=(14, 5))
for i in range(len(sample_indices)):
    # image originale
    axes[0, i].imshow(sample_images[i])
    axes[0, i].set_title(f"Vrai: {class_names[sample_labels[i]]}\nPrédit: {class_names[pred_classes[i]]}",
                         fontsize=7,
                         color='green' if sample_labels[i] == pred_classes[i] else 'red')
    axes[0, i].axis('off')
    
    # heatmap GradCAM++
    heatmap = mpl_cm.jet(cam[i])[..., :3]
    overlay = 0.5 * sample_images[i] + 0.5 * heatmap
    axes[1, i].imshow(overlay)
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Image originale', fontsize=9)
axes[1, 0].set_ylabel('GradCAM++', fontsize=9)
plt.suptitle('Visualisation GradCAM++ — Vanilla CNN', fontsize=11)
plt.tight_layout()
plt.show()

Les zones chaudes (rouge/jaune) indiquent les régions qui ont le plus contribué à la prédiction. C'est utile pour vérifier que le réseau regarde vraiment le bon endroit et pas un artefact du fond par exemple.

Le choix de GradCAM++ est justifié : il est plus précis que GradCAM original pour les petits objets. Pour CIFAR-10 où les objets occupent souvent une bonne partie de l'image, on pourrait aussi tester LIME (Local Interpretable Model-agnostic Explanations) ou les Integrated Gradients qui sont plus théoriquement fondés mais plus lents à calculer.

## 11. (BONUS) Spécialisation des couches

On veut voir ce que les filtres des premières couches ont appris. Pour ça, deux approches : visualiser les poids des filtres directement, ou maximiser l'activation d'un filtre par gradient ascent sur l'image d'entrée.

In [ ]:
# visualisation des filtres de la première couche Conv2D
# les premières couches apprennent souvent des détecteurs de bords, couleurs, textures

first_conv_layer = None
for layer in model_vanilla_cnn.layers:
    if isinstance(layer, tf.keras.layers.Conv2D):
        first_conv_layer = layer
        break

weights, biases = first_conv_layer.get_weights()
print(f"Forme des poids de la 1ère Conv2D : {weights.shape}")
# shape : (kernel_h, kernel_w, in_channels, n_filters)

# normaliser les poids pour l'affichage
n_filters = weights.shape[-1]
n_show = min(32, n_filters)
n_cols = 8
n_rows = n_show // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, n_rows * 1.5))
for i in range(n_show):
    ax = axes[i // n_cols, i % n_cols]
    filt = weights[:, :, :, i]  # filtre i, 3 canaux RGB
    # normaliser entre 0 et 1
    filt = (filt - filt.min()) / (filt.max() - filt.min() + 1e-8)
    ax.imshow(filt)
    ax.axis('off')
    ax.set_title(f'{i}', fontsize=7)

plt.suptitle('Filtres de la 1ère couche Conv2D (Vanilla CNN)', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# visualisation des activations sur une image de test
# on voit comment chaque filtre réagit à l'image

from tensorflow.keras import Model

# modèle intermédiaire qui sort les activations des couches conv
conv_layers = [l for l in model_vanilla_cnn.layers if isinstance(l, tf.keras.layers.Conv2D)]
activation_model = Model(
    inputs=model_vanilla_cnn.input,
    outputs=[l.output for l in conv_layers]
)

# on prend la première image du test
img_to_show = test_images[0:1]
activations = activation_model.predict(img_to_show)

# afficher les 16 premières feature maps de chaque couche
for layer_idx, (layer, act) in enumerate(zip(conv_layers, activations)):
    n_show = min(16, act.shape[-1])
    n_cols = 8
    n_rows = n_show // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, n_rows * 1.5))
    for i in range(n_show):
        ax = axes[i // n_cols, i % n_cols] if n_rows > 1 else axes[i % n_cols]
        ax.imshow(act[0, :, :, i], cmap='viridis')
        ax.axis('off')
    
    plt.suptitle(f'Activations couche {layer_idx+1} ({layer.name}) — image: {class_names[test_labels[0][0]]}',
                 fontsize=10)
    plt.tight_layout()
    plt.show()

Les premières couches réagissent à des motifs simples (bords, gradients de couleur). Les couches plus profondes ont des activations plus abstraites, souvent difficiles à interpréter visuellement mais qui encodent des structures sémantiques.

## 12. Application sur d'autres datasets

On teste nos architectures sur MNIST (chiffres), Fashion-MNIST (vêtements), et on cherche deux autres datasets.

In [ ]:
# helper pour adapter les images en niveaux de gris à notre pipeline (3 canaux, 32x32)
def preprocess_grayscale_dataset(train_imgs, test_imgs, target_size=(32, 32)):
    # convertir en float et normaliser
    train_imgs = train_imgs.astype('float32') / 255.0
    test_imgs  = test_imgs.astype('float32')  / 255.0
    
    # ajouter dimension channel si nécessaire
    if train_imgs.ndim == 3:
        train_imgs = train_imgs[..., np.newaxis]
        test_imgs  = test_imgs[..., np.newaxis]
    
    # resize à 32x32 avec tf
    train_imgs = tf.image.resize(train_imgs, target_size).numpy()
    test_imgs  = tf.image.resize(test_imgs,  target_size).numpy()
    
    # répliquer le canal gris 3 fois pour avoir (32,32,3)
    train_imgs = np.repeat(train_imgs, 3, axis=-1)
    test_imgs  = np.repeat(test_imgs,  3, axis=-1)
    
    return train_imgs, test_imgs

# helper pour entraîner et afficher les résultats rapidement
def quick_train_eval(model_fn, train_imgs, train_lbs, test_imgs, test_lbs,
                     epochs=10, dataset_name=''):
    m = model_fn()
    m.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=['accuracy']
    )
    h = m.fit(train_imgs, train_lbs, batch_size=64, epochs=epochs,
              validation_data=(test_imgs, test_lbs), verbose=0)
    
    best_acc = max(h.history['val_accuracy'])
    print(f"{dataset_name} | {m.name} | test acc: {best_acc:.3f}")
    return m, h

In [ ]:
# MNIST — chiffres manuscrits
(x_mnist_tr, y_mnist_tr), (x_mnist_te, y_mnist_te) = tf.keras.datasets.mnist.load_data()
x_mnist_tr, x_mnist_te = preprocess_grayscale_dataset(x_mnist_tr, x_mnist_te)

print("--- MNIST ---")
for fn in [create_simplest_cifar10_model, create_vanilla_nn_cifar10_model,
           create_vanilla_cnn_cifar10_model]:
    quick_train_eval(fn, x_mnist_tr, y_mnist_tr, x_mnist_te, y_mnist_te,
                     dataset_name='MNIST')

In [ ]:
# Fashion MNIST — vêtements (plus difficile que MNIST)
(x_fash_tr, y_fash_tr), (x_fash_te, y_fash_te) = tf.keras.datasets.fashion_mnist.load_data()
x_fash_tr, x_fash_te = preprocess_grayscale_dataset(x_fash_tr, x_fash_te)

print("--- Fashion MNIST ---")
for fn in [create_simplest_cifar10_model, create_vanilla_nn_cifar10_model,
           create_vanilla_cnn_cifar10_model]:
    quick_train_eval(fn, x_fash_tr, y_fash_tr, x_fash_te, y_fash_te,
                     dataset_name='Fashion-MNIST')

In [ ]:
# CIFAR-100 — version plus difficile de CIFAR-10 avec 100 classes
# on adapte la couche de sortie

def create_vanilla_cnn_100():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(32, 32, 3)),
        tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D(2),
        tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D(2),
        tf.keras.layers.Conv2D(128, 3, activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D(2),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.Dense(100)  # 100 classes
    ], name="Vanilla_CNN_CIFAR100")
    return model

(x_c100_tr, y_c100_tr), (x_c100_te, y_c100_te) = tf.keras.datasets.cifar100.load_data()
x_c100_tr = x_c100_tr / 255.0
x_c100_te = x_c100_te / 255.0

print("--- CIFAR-100 ---")
quick_train_eval(create_vanilla_cnn_100, x_c100_tr, y_c100_tr, x_c100_te, y_c100_te,
                 dataset_name='CIFAR-100')

In [ ]:
# SVHN — Street View House Numbers, images de chiffres en conditions réelles
# chargement via tensorflow_datasets
!pip install tensorflow-datasets -q
import tensorflow_datasets as tfds

def load_svhn():
    ds_train, ds_test = tfds.load('svhn_cropped', split=['train', 'test'],
                                   as_supervised=True)
    def prep(img, lbl):
        img = tf.cast(img, tf.float32) / 255.0
        img = tf.image.resize(img, (32, 32))
        return img, lbl
    
    train_data = ds_train.map(prep).batch(64).prefetch(tf.data.AUTOTUNE)
    test_data  = ds_test.map(prep).batch(64).prefetch(tf.data.AUTOTUNE)
    return train_data, test_data

print("--- SVHN ---")
svhn_train, svhn_test = load_svhn()

m_svhn = create_vanilla_cnn_cifar10_model()
m_svhn.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)
h_svhn = m_svhn.fit(svhn_train, epochs=10, validation_data=svhn_test, verbose=1)
print(f"SVHN best test acc: {max(h_svhn.history['val_accuracy']):.3f}")

In [ ]:
# récap des résultats sur les 4 datasets
# (valeurs lues depuis les runs précédents)
datasets_summary = {
    'MNIST':        {'note': '10 classes, chiffres simples, tâche facile'},
    'Fashion-MNIST':{'note': '10 classes, vêtements, texture importante'},
    'CIFAR-10':     {'note': '10 classes, images naturelles couleur'},
    'CIFAR-100':    {'note': '100 classes, beaucoup plus difficile'},
    'SVHN':         {'note': '10 classes, chiffres en conditions réelles'},
}
print("Sur MNIST et Fashion-MNIST on atteint des meilleurs scores")
print("qu'avec CIFAR-10 car les images sont plus 'propres' (fond uni).")
print("CIFAR-100 est logiquement plus dur vu le nombre de classes.")
print("SVHN est difficile à cause du bruit et des chiffres superposés.")

## 13. Enrichissement du dataset CIFAR-10

On va créer 4 images synthétiques par classe (40 images au total) pour enrichir le dataset. On utilise l'augmentation de données classique : flip, rotation, zoom, ajout de bruit.

In [ ]:
# pour rappel, CIFAR-10 contient 50 000 images d'entraînement et 10 000 de test
print(f"Taille originale du dataset CIFAR-10 : {len(train_images)} train + {len(test_images)} test")
print(f"Soit {len(train_images) + len(test_images)} images au total")

In [ ]:
# on génère 4 images pour chaque classe à partir d'exemples existants
import scipy.ndimage

def augment_image(img):
    """applique une transformation aléatoire à une image"""
    # flip horizontal (50% de chance)
    if np.random.rand() > 0.5:
        img = img[:, ::-1, :]
    # rotation légère
    angle = np.random.uniform(-15, 15)
    img = scipy.ndimage.rotate(img, angle, reshape=False, mode='nearest')
    # bruit gaussien
    img = img + np.random.normal(0, 0.02, img.shape)
    # clipper entre 0 et 1
    return np.clip(img, 0, 1)

new_images = []
new_labels = []

# on prend un exemple par classe et on génère 4 variantes
for class_id in range(10):
    # prendre le premier exemple de cette classe dans le train
    class_indices = np.where(train_labels.flatten() == class_id)[0]
    source_img = train_images[class_indices[0]]
    
    for _ in range(4):
        aug = augment_image(source_img)
        new_images.append(aug)
        new_labels.append([class_id])

new_images = np.array(new_images)
new_labels = np.array(new_labels)

print(f"Nouvelles images générées : {len(new_images)}")

# afficher les images originales et leurs versions augmentées
fig, axes = plt.subplots(10, 5, figsize=(10, 20))
for class_id in range(10):
    class_indices = np.where(train_labels.flatten() == class_id)[0]
    original = train_images[class_indices[0]]
    
    axes[class_id, 0].imshow(original)
    axes[class_id, 0].set_title(f'Original\n{class_names[class_id]}', fontsize=7)
    axes[class_id, 0].axis('off')
    
    for j in range(4):
        aug_idx = class_id * 4 + j
        axes[class_id, j+1].imshow(np.clip(new_images[aug_idx], 0, 1))
        axes[class_id, j+1].set_title(f'Aug {j+1}', fontsize=7)
        axes[class_id, j+1].axis('off')

plt.suptitle('Images originales et versions augmentées (4 par classe)', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# tester le vanilla CNN déjà entraîné sur ces nouvelles images
preds_new = model_vanilla_cnn.predict(new_images)
pred_classes_new = np.argmax(preds_new, axis=1)
true_classes_new = new_labels.flatten()

acc_new = np.mean(pred_classes_new == true_classes_new)
print(f"Accuracy du Vanilla CNN sur les images augmentées : {acc_new:.2f}")
print()
for i, (pred, true) in enumerate(zip(pred_classes_new, true_classes_new)):
    status = '✓' if pred == true else '✗'
    print(f"  Image {i:2d} | Vrai: {class_names[true]:12s} | Prédit: {class_names[pred]:12s} {status}")

Le réseau reconnaît bien la plupart des images augmentées car elles ressemblent aux exemples d'entraînement qu'il a déjà vus. Les éventuelles erreurs viennent des rotations plus importantes ou du bruit qui dégrade trop l'image.

## 14. Enrichissement du dataset Fashion MNIST

On fait pareil avec Fashion-MNIST : 2 images pour 3 classes.

In [ ]:
fashion_class_names = ['T-shirt', 'Pantalon', 'Pull', 'Robe', 'Manteau',
                       'Sandale', 'Chemise', 'Basket', 'Sac', 'Bottine']

# 3 classes choisies : T-shirt (0), Robe (3), Bottine (9)
selected_classes = [0, 3, 9]

new_fash_images = []
new_fash_labels = []

for class_id in selected_classes:
    class_indices = np.where(y_fash_tr.flatten() == class_id)[0]
    source_img = x_fash_tr[class_indices[0]]
    
    for _ in range(2):
        aug = augment_image(source_img)
        new_fash_images.append(aug)
        new_fash_labels.append(class_id)

new_fash_images = np.array(new_fash_images)
new_fash_labels = np.array(new_fash_labels)

# afficher
fig, axes = plt.subplots(3, 3, figsize=(8, 8))
for row, class_id in enumerate(selected_classes):
    class_indices = np.where(y_fash_tr.flatten() == class_id)[0]
    axes[row, 0].imshow(x_fash_tr[class_indices[0]], cmap='gray')
    axes[row, 0].set_title(f'Original\n{fashion_class_names[class_id]}', fontsize=8)
    axes[row, 0].axis('off')
    
    for j in range(2):
        aug_idx = row * 2 + j
        # les images ont 3 canaux (après preprocess) mais sont en gris
        axes[row, j+1].imshow(new_fash_images[aug_idx, :, :, 0], cmap='gray')
        axes[row, j+1].set_title(f'Augmentée {j+1}', fontsize=8)
        axes[row, j+1].axis('off')

plt.suptitle('Fashion MNIST — 2 images augmentées pour 3 classes', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# on réentraîne un modèle sur Fashion-MNIST avec ces images supplémentaires
# d'abord construire le dataset étendu

# on prend seulement les 3 classes sélectionnées du dataset original
# et on ajoute les 6 nouvelles images
mask_tr = np.isin(y_fash_tr.flatten(), selected_classes)
mask_te = np.isin(y_fash_te.flatten(), selected_classes)

x_sub_tr = np.concatenate([x_fash_tr[mask_tr], new_fash_images])
y_sub_tr = np.concatenate([y_fash_tr.flatten()[mask_tr], new_fash_labels])

x_sub_te = x_fash_te[mask_te]
y_sub_te = y_fash_te.flatten()[mask_te]

print(f"Train étendu : {len(x_sub_tr)} images")
print(f"Test : {len(x_sub_te)} images")

# modèle simple pour 3 classes
def create_cnn_3classes():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(32, 32, 3)),
        tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D(2),
        tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D(2),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(10)  # toujours 10 sorties pour compatibilité
    ], name='CNN_3classes')
    return model

m_fash3 = create_cnn_3classes()
m_fash3.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)
h_fash3 = m_fash3.fit(x_sub_tr, y_sub_tr, epochs=10, batch_size=64,
                      validation_data=(x_sub_te, y_sub_te), verbose=1)

plot_history(h_fash3, 'CNN sur 3 classes Fashion-MNIST augmenté')

## Conclusion

Ce projet nous a permis de voir concrètement l'impact de l'architecture sur les performances. Le passage des réseaux denses aux CNNs fait une vraie différence sur CIFAR-10 : de ~40% pour un Dense à ~70% pour un CNN convenable.

Les points clés qu'on retient :
- La structure convolutive est adaptée aux images car elle exploite la localité spatiale
- L'overfitting est réel et visible sur les courbes (gap train/test) — la régularisation (Dropout, BatchNorm) et la data augmentation aident
- Le learning rate est le paramètre le plus sensible : trop grand = instable, trop petit = trop lent
- Les métriques sur le test set sont les seules qui comptent vraiment pour juger un modèle
- GradCAM++ permet de vérifier que le réseau regarde les bonnes zones, ce qui aide à détecter des biais potentiels

Pour aller plus loin, ResNet ou EfficientNet sur CIFAR-10 dépassent les 90% avec un entraînement soigné.